In [5]:
import pandas as pd
import numpy as np
import time
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [6]:
import importlib
import feature_engineer
importlib.reload(feature_engineer)

<module 'feature_engineer' from '/Users/conan/Lucas_Systems_Capstone_Project/Model_Jiashen/feature_engineer.py'>

In [15]:
# Configurations
warehouses = ["OE", "OF"]
work_codes = ["10", "20", "30"]
max_time = 300
models = []
results = []

In [16]:
from feature_engineer import get_engineered_df

for wh in warehouses:
    DATA_PATH = f"../data/processed/{wh.lower()}_detailed.parquet"
    for wc in work_codes:
        print(f"--- Training Model for {wh} | WC {wc} ---")
        
        # 1. Preprocess
        df, features, cat_cols = get_engineered_df(DATA_PATH, warehouse=wh, work_code=wc, max_time=max_time)
        med = np.median(df["Time_Delta_sec"])
        avg = np.mean(df["Time_Delta_sec"])
        
        # 2. Prepare Data (One-Hot Encoding)
        X = pd.get_dummies(df[features], columns=cat_cols, drop_first=True)
        y = df["Time_Delta_sec"]
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        # 3. Model Training & Timing
        start_time = time.time()
        model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        end_time = time.time()
        models.append(model)
        
        # 4. Evaluation
        preds = model.predict(X_test)
        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        runtime = end_time - start_time
        
        # 5. Store Results
        results.append({
            "Warehouse": wh,
            "WorkCode": wc,
            "Train_Rows": len(X_train),
            "R^2": round(r2, 4),
            "MAE": round(mae, 2),
            "Runtime_Sec": round(runtime, 2),
            "Median Task Time": med,
            "Average Task Time": avg
        })

# Display Clean Results Table
results_df = pd.DataFrame(results)
display(results_df)

--- Training Model for OE | WC 10 ---
--- Training Model for OE | WC 20 ---
--- Training Model for OE | WC 30 ---
--- Training Model for OF | WC 10 ---
--- Training Model for OF | WC 20 ---
--- Training Model for OF | WC 30 ---


,Warehouse,WorkCode,Train_Rows,R^2,MAE,Runtime_Sec,Median Task Time,Average Task Time
0,OE,10,3263,0.2962,35.01,1.28,68.4930,84.905957
1,OE,20,17012,0.6471,17.06,1.34,0.4670,36.392604
2,OE,30,52238,0.3955,22.07,1.66,33.0070,45.876832
3,OF,10,3740,0.4312,30.10,1.20,52.1685,65.207993
4,OF,20,32110,0.6400,16.83,1.41,0.5000,33.092137
5,OF,30,60366,0.4056,27.78,1.71,45.5870,59.876979
